# Spam Detection with Naive Bayes

**CS 212 AI Programming 1**

This notebook trains a *naive Bayes* classifier to sort text messages into **spam** (unwanted ads and scams) and **ham** (normal messages). It uses the same scikit-learn tools as the [Working With Text Data](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html) tutorial: `CountVectorizer` to turn text into word counts and `MultinomialNB` to classify it.

The steps are:

1. Load and explore the data.
2. Split the data into a training set and a test set.
3. Turn the text into word counts (feature vectors).
4. Train the classifier.
5. Test how accurate it is.
6. Classify some new messages.
7. See which words the model thinks are the strongest signs of spam.

This notebook runs in VS Code or in Google Colab.

## 1. Load and Explore the Data

The data is the [SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection) from the UCI Machine Learning Repository. It has 5,574 real text messages from the UK and Singapore. Each one was labeled by a person as `ham` or `spam`.

We'll use the `pandas` library to read the data file. It has two columns: `label` and `message`.

In [1]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/LCC-CIT/CS212-CourseMaterials/main/docs/Examples/SpamDetection/sms_spam.csv"
messages = pd.read_csv(DATA_URL)
messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


`read_csv` loads the file into a pandas `DataFrame`, which is a table of rows and columns. `head()` shows the first five rows.

How many messages of each kind are there?

In [2]:
messages["label"].value_counts()

label
ham     4827
spam     747
Name: count, dtype: int64

Only about 13% of the messages are spam. That's worth remembering: a "classifier" that labeled *every* message as ham would be about 87% accurate without learning anything. Our model needs to do better than that.

## 2. Split the Data into Training and Test Sets

We train the model on one part of the data and test it on messages it has never seen. `train_test_split` shuffles the messages and holds back 25% of them for testing. Setting `random_state` makes the shuffle the same every time you run the notebook.

In [3]:
from sklearn.model_selection import train_test_split

train_text, test_text, train_labels, test_labels = train_test_split(
    messages["message"], messages["label"], test_size=0.25, random_state=42)

print("Training messages:", len(train_text))
print("Test messages:", len(test_text))

Training messages: 4180
Test messages: 1394


## 3. Turn the Text into Word Counts

A classifier needs numbers, not text. `CountVectorizer` builds a dictionary of every word in the training messages, then turns each message into a row of counts: how many times each word appears in it. (This is the *bag of words* from the Working With Text Data tutorial.)

- `fit_transform` learns the dictionary from the training messages *and* counts their words.
- `transform` counts the words in the test messages using that same dictionary. Words that weren't in the training messages are ignored.

In [4]:
from sklearn.feature_extraction.text import CountVectorizer

count_vect = CountVectorizer()
train_counts = count_vect.fit_transform(train_text)
test_counts = count_vect.transform(test_text)

train_counts.shape

(4180, 7462)

The result has one row per training message and one column per word in the dictionary.

## 4. Train the Classifier

`MultinomialNB` is the naive Bayes classifier made for word counts. Training (`fit`) is very fast: for each word, it calculates how often that word appears in spam messages and how often it appears in ham messages.

In [5]:
from sklearn.naive_bayes import MultinomialNB

clf = MultinomialNB()
clf.fit(train_counts, train_labels)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[3624., 556.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.14,-2.02]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[<U4](2,)","['ham','spam']"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 7462)","[[ 0., 0., 1.,..., 0., 1., 1.], [ 7.,22., 0.,..., 1., 0., 0.]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 7462)","[[-10.91,-10.91,-10.22,...,-10.91,-10.22,-10.22], [ -7.85, -6.79, -9.93,..., -9.24, -9.93, -9.93]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,7462


## 5. Test the Classifier

Now we ask the model to classify the test messages, and compare its answers with the real labels.

In [6]:
from sklearn import metrics

predicted = clf.predict(test_counts)
print("Accuracy:", metrics.accuracy_score(test_labels, predicted))

Accuracy: 0.9863701578192252


That's much better than the 87% we'd get by calling everything ham. The classification report shows how well the model does on each kind of message:

- **Precision** for spam: of the messages the model *called* spam, how many really were spam.
- **Recall** for spam: of the messages that really *were* spam, how many the model caught.

In [7]:
print(metrics.classification_report(test_labels, predicted))

              precision    recall  f1-score   support

         ham       0.99      1.00      0.99      1203
        spam       0.97      0.93      0.95       191

    accuracy                           0.99      1394
   macro avg       0.98      0.96      0.97      1394
weighted avg       0.99      0.99      0.99      1394



The *confusion matrix* shows the same thing as counts. Rows are the real labels and columns are the model's predictions, both in the order ham, spam.

In [8]:
metrics.confusion_matrix(test_labels, predicted)

array([[1197,    6],
       [  13,  178]])

|                 | Predicted ham                          | Predicted spam                      |
| --------------- | -------------------------------------- | ----------------------------------- |
| **Actual ham**  | ham correctly let through              | ham wrongly sent to the spam folder |
| **Actual spam** | spam that got past the filter          | spam correctly caught               |

For a spam filter, sending a real message to the spam folder is usually worse than letting a spam message through, so the top-right number is the one to watch.

## 6. Classify New Messages

To classify new messages, we turn them into word counts with `transform` (not `fit_transform`, since the dictionary was already built from the training data), then call `predict`.

`predict_proba` gives the model's probability for each label. The columns are in the order of `clf.classes_`, which is `['ham', 'spam']`, so column 1 is the probability of spam.

Try adding your own messages to the list!

In [9]:
new_messages = [
    "Hey, are we still meeting for lunch tomorrow?",
    "WINNER! You have won a free cruise. Call now to claim your prize!",
    "Can you send me the notes from today's class?",
    "URGENT: your account has been selected for a cash reward. Text CLAIM to 80082",
]

new_counts = count_vect.transform(new_messages)
new_predictions = clf.predict(new_counts)
new_probabilities = clf.predict_proba(new_counts)

for i in range(len(new_messages)):
    spam_chance = new_probabilities[i][1]
    print(f"{new_predictions[i]:4}  ({spam_chance:.1%} spam)  {new_messages[i]}")

ham   (0.0% spam)  Hey, are we still meeting for lunch tomorrow?
spam  (100.0% spam)  WINNER! You have won a free cruise. Call now to claim your prize!
ham   (0.0% spam)  Can you send me the notes from today's class?
spam  (100.0% spam)  URGENT: your account has been selected for a cash reward. Text CLAIM to 80082


## 7. Which Words Signal Spam?

Naive Bayes works by comparing, for each word, how likely it is to appear in spam versus in ham. Dividing those two probabilities gives the word's **likelihood ratio**, just like the likelihood ratios in the lecture notes on Bayes' rule.

The trained classifier stores these probabilities in `feature_log_prob_` (as logarithms, so we use `np.exp` to turn them back into probabilities). Row 0 is ham and row 1 is spam.

In [10]:
import numpy as np

words = count_vect.get_feature_names_out()
ham_probs = np.exp(clf.feature_log_prob_[0])
spam_probs = np.exp(clf.feature_log_prob_[1])
ratios = spam_probs / ham_probs

# argsort lists the positions of the ratios from smallest to largest,
# so we take the last 10 positions and go through them backward.
top_ten = np.argsort(ratios)[-10:]
for i in reversed(top_ten):
    print(f"{words[i]:10} {ratios[i]:6.1f} times more likely in spam")

claim       221.6 times more likely in spam
prize       168.2 times more likely in spam
150p        146.9 times more likely in spam
tone        125.5 times more likely in spam
guaranteed  104.1 times more likely in spam
500          93.5 times more likely in spam
18           93.5 times more likely in spam
cs           90.8 times more likely in spam
1000         88.1 times more likely in spam
awarded      82.8 times more likely in spam


Words like "claim" and "prize" show up far more often in spam. So do numbers, such as "150p" (a price of 150 pence, since many of these messages are from the UK) and "18" (as in "18+ only"). When a message contains several of these words, the classifier multiplies their likelihood ratios together, and the odds of spam quickly become very high.

---

**Data source:** Almeida, T.A., Gómez Hidalgo, J.M., and Yamakami, A. "Contributions to the Study of SMS Spam Filtering: New Collection and Results." *Proceedings of the 2011 ACM Symposium on Document Engineering (DOCENG'11)*, 2011. Downloaded from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/228/sms+spam+collection). The data file's original readme, including its license, is in `SMSSpamCollection-readme.txt`.

This notebook was drafted by Claude Opus 5.5.